# Unified Spatial State, District, and Region Mapping
This notebook performs **100% unified geospatial mapping** for all 5,933 cities in  using a single GeoPandas spatial Point-in-Polygon join against official Indian District boundary geometries.

In [30]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# 1. Load Dataset & Official Indian District Boundaries
csv_path = '../../tempdata/weather.csv'
df = pd.read_csv(csv_path)

for col in ['state', 'district', 'region', 'Unnamed: 0']:
    if col in df.columns:
        df = df.drop(columns=[col])

# Single GeoJSON file containing official State (NAME_1) and District (NAME_2) polygons
url = 'https://raw.githubusercontent.com/geohacker/india/master/district/india_district.geojson'
gdf_districts = gpd.read_file(url)

# 2. Single Unified GeoPandas Spatial Join (State & District simultaneously)
geometry = [Point(xy) for xy in zip(df['lng'], df['lat'])]
gdf_cities = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

# Point-in-Polygon Spatial Join
joined = gpd.sjoin(gdf_cities, gdf_districts[['NAME_1', 'NAME_2', 'geometry']], how='left', predicate='intersects')

# Distance-based nearest polygon match for coastal boundary edge points
unmapped_mask = joined['NAME_1'].isnull() | joined['NAME_2'].isnull()
if unmapped_mask.any():
    unmapped_cities = gdf_cities[unmapped_mask].to_crs('EPSG:3857')
    districts_proj = gdf_districts.to_crs('EPSG:3857')
    nearest = gpd.sjoin_nearest(unmapped_cities, districts_proj[['NAME_1', 'NAME_2', 'geometry']], how='left')
    joined.loc[unmapped_mask, 'NAME_1'] = nearest['NAME_1']
    joined.loc[unmapped_mask, 'NAME_2'] = nearest['NAME_2']

# Standardize State Names
joined['NAME_1'] = joined['NAME_1'].replace({
    'Orissa': 'Odisha',
    'Uttaranchal': 'Uttarakhand',
    'Andaman and Nicobar': 'Andaman & Nicobar',
    'Jammu and Kashmir': 'Jammu & Kashmir'
})

# Insert State & District Columns
df.insert(df.columns.get_loc('city') + 1, 'state', joined['NAME_1'].tolist())
df.insert(df.columns.get_loc('state') + 1, 'district', joined['NAME_2'].tolist())

# 3. Map Region
region_mapping = {
    'Jammu & Kashmir': 'Northern', 'Jammu and Kashmir': 'Northern', 'Ladakh': 'Northern',
    'Himachal Pradesh': 'Northern', 'Punjab': 'Northern', 'Chandigarh': 'Northern',
    'Uttarakhand': 'Northern', 'Uttaranchal': 'Northern', 'Haryana': 'Northern',
    'Delhi': 'Northern', 'Rajasthan': 'Northern',

    'Uttar Pradesh': 'North Central',
    'Madhya Pradesh': 'Central', 'Chhattisgarh': 'Central',

    'Bihar': 'Eastern', 'Jharkhand': 'Eastern', 'West Bengal': 'Eastern',
    'Odisha': 'Eastern', 'Orissa': 'Eastern',

    'Assam': 'Northeast', 'Sikkim': 'Northeast', 'Arunachal Pradesh': 'Northeast',
    'Nagaland': 'Northeast', 'Manipur': 'Northeast', 'Mizoram': 'Northeast',
    'Tripura': 'Northeast', 'Meghalaya': 'Northeast',

    'Gujarat': 'Western', 'Maharashtra': 'Western', 'Goa': 'Western',
    'Dadra and Nagar Haveli and Daman and Diu': 'Western',
    'Dadra and Nagar Haveli': 'Western', 'Daman and Diu': 'Western',

    'Andhra Pradesh': 'Southern', 'Telangana': 'Southern', 'Karnataka': 'Southern',
    'Tamil Nadu': 'Southern', 'Kerala': 'Southern', 'Puducherry': 'Southern',
    'Lakshadweep': 'Southern', 'Andaman & Nicobar': 'Southern', 'Andaman and Nicobar': 'Southern'
}

df.insert(df.columns.get_loc('district') + 1, 'region', df['state'].map(region_mapping))

# 4. Save Master Lookup File
df.to_csv(csv_path, index=False)
print("Successfully generated Master Lookup File using Single Unified Spatial Method!")


Successfully generated Master Lookup File using Single Unified Spatial Method!
